---

# Phase 9 · Section 2 — API Layer

### Objective

Phase 9 Section 2 adds a **clean, documented REST API** to the portfolio — exposing project data through structured endpoints that return consistent JSON responses.

Phase 9 Section 1 established the admin-facing architecture: authenticated routes, project CRUD, and a full taxonomy management system. The portfolio is now maintainable without touching the codebase.

Phase 9 Section 2 adds the public data layer.

A REST API is not a feature the portfolio needs to function — it is a deliberate architectural statement. It demonstrates that the application is designed with separation of concerns in mind: data lives in one place, presentation in another, and the boundary between them is clearly defined. That is mid-level thinking, and it shows.

This section covers three deliverables:

* REST endpoints — structured routes for retrieving project data individually and as collections
* JSON responses — consistent, well-shaped output with appropriate status codes and error handling
* Front-end decoupling foundation — an architecture that would support a React or other front end without back-end changes

By the end of this section, the portfolio's data is accessible through a documented public API — a concrete proof signal that goes beyond what most junior portfolios ever attempt.

---

## Prerequisites

The following must be in place before beginning Section 2:

* Fully deployed application with public URL — from Phase 7 Section 2
* Projects stored in a relational database with category and tag relationships — from Phase 8
* Admin dashboard operational — from Phase 9 Section 1
* Stable production environment

Section 2 introduces a new Flask blueprint. No existing routes or templates are modified.

---

## Implementation Steps

1. Register a dedicated `api` blueprint at `/api/v1`
2. Implement the projects collection endpoint
3. Implement the single project endpoint
4. Implement supporting endpoints for categories and tags
5. Standardise the JSON response envelope across all endpoints
6. Add error handlers for 404 and 500 within the API blueprint
7. Validate all endpoints return correct data and status codes

---

# Role of the API Layer

## Purpose

The API layer exists to **separate data from presentation**. The Jinja2 templates that render the public portfolio are one consumer of the project data — but they should not be the only possible consumer.

By exposing the data through a REST API, the portfolio gains:

* **Architectural clarity** — the application knows the difference between its data layer and its presentation layer
* **Extensibility** — a future React front end, a mobile view, or an external integration can consume the same data without changes to the back end
* **A proof signal** — a working, versioned REST API is something a junior portfolio almost never has, and something a mid-level role will expect you to understand

---

## What This Section Delivers

This section implements three things:

```text
1. REST Endpoints     — structured routes for projects, categories, and tags
2. JSON Responses     — consistent envelope, predictable error shapes, correct status codes
3. Decoupling Layer   — an architecture ready for front-end separation without back-end rewrites
```

Each is treated as a production-grade feature, not a prototype.

---

# 2.1 — REST Endpoints for Projects

## Purpose

The project data is the core content of the portfolio. It needs to be accessible in a structured, predictable way — both as a collection and as individual records.

A REST endpoint for projects is also a direct answer to the question any technical interviewer will ask: *"Have you ever designed an API?"*

---

## Endpoint Design

The API should be versioned from the start. All routes live under `/api/v1/`.

| Method | Endpoint | Description |
|---|---|---|
| GET | `/api/v1/projects` | All published projects |
| GET | `/api/v1/projects/<slug>` | Single project by slug |
| GET | `/api/v1/projects?category=<slug>` | Projects filtered by category |
| GET | `/api/v1/projects?featured=true` | Featured projects only |
| GET | `/api/v1/categories` | All categories |
| GET | `/api/v1/tags` | All tags with project counts |

All endpoints are read-only. The API is public — no authentication required to consume it. Write operations remain admin-only and are not exposed through the API.

---

## Blueprint Registration

The API is implemented as a standalone Flask blueprint to keep it cleanly separated from the main application routes:

```python
# app/api/__init__.py
from flask import Blueprint

api_bp = Blueprint('api', __name__, url_prefix='/api/v1')

from app.api import routes
```

```python
# app/app.py
from app.api import api_bp
app.register_blueprint(api_bp)
```

---

## Project Collection Endpoint

```python
@api_bp.route('/projects')
def get_projects():
    category = request.args.get('category')
    featured = request.args.get('featured')

    query = Project.query.filter_by(status='published')

    if category:
        query = query.join(Category).filter(Category.slug == category)
    if featured:
        query = query.filter_by(featured=True)

    projects = query.order_by(Project.created_at.desc()).all()
    return jsonify(success_response([p.to_dict() for p in projects]))
```

---

## Single Project Endpoint

```python
@api_bp.route('/projects/<slug>')
def get_project(slug):
    project = Project.query.filter_by(slug=slug, status='published').first()
    if not project:
        return jsonify(error_response('Project not found')), 404
    return jsonify(success_response(project.to_dict()))
```

---

## Model Serialisation

Each model requires a `to_dict()` method to control exactly what is included in API responses. Sensitive fields, internal IDs, and admin-only data should not be exposed:

```python
# app/models/project.py
def to_dict(self):
    return {
        'title': self.title,
        'slug': self.slug,
        'summary': self.summary,
        'description': self.description,
        'category': self.category.name if self.category else None,
        'tags': [tag.name for tag in self.tags],
        'github_url': self.github_url,
        'live_url': self.live_url,
        'featured': self.featured,
        'card_image': self.card_image,
        'screenshots': self.screenshots or [],
        'created_at': self.created_at.isoformat() if self.created_at else None,
    }
```

---

## Files Involved

```bash
app/api/__init__.py             # Blueprint definition
app/api/routes.py               # All API route handlers
app/models/project.py           # to_dict() method
app/models/category.py          # to_dict() method
app/models/tag.py               # to_dict() method
```

---

## Validation Checklist

* `GET /api/v1/projects` returns all published projects as a JSON array
* `GET /api/v1/projects/<slug>` returns a single project or a 404
* `GET /api/v1/projects?category=web` returns only projects in that category
* `GET /api/v1/projects?featured=true` returns only featured projects
* `GET /api/v1/categories` returns all categories
* `GET /api/v1/tags` returns all tags with project counts
* Draft projects are never returned by any endpoint

---

# 2.2 — JSON Responses

## Purpose

A working API is not enough. The responses need to be consistent — same shape, same field names, same error format — across every endpoint. Inconsistent API responses are a signal that the builder has not thought about the interface as a contract.

Consistency here is the difference between an API that was hacked together and one that was designed.

---

## Response Envelope

All responses — success and error — should share a consistent outer shape:

```python
# Success
{
    "success": true,
    "data": { ... }       # object for single resource
}

# Success — collection
{
    "success": true,
    "data": [ ... ],      # array for collections
    "count": 12
}

# Error
{
    "success": false,
    "error": "Project not found"
}
```

---

## Helper Functions

Define envelope helpers once and use them across all routes:

```python
# app/api/utils.py

def success_response(data):
    if isinstance(data, list):
        return {'success': True, 'data': data, 'count': len(data)}
    return {'success': True, 'data': data}

def error_response(message):
    return {'success': False, 'error': message}
```

---

## Status Codes

| Situation | Status code |
|---|---|
| Successful GET | 200 |
| Resource not found | 404 |
| Server error | 500 |

The API does not accept POST, PUT, or DELETE requests — no 201 or 204 responses are required.

---

## Error Handlers

Register error handlers scoped to the API blueprint so that JSON errors are returned rather than HTML pages:

```python
@api_bp.errorhandler(404)
def not_found(e):
    return jsonify(error_response('Resource not found')), 404

@api_bp.errorhandler(500)
def server_error(e):
    return jsonify(error_response('Internal server error')), 500
```

Without these, Flask will return an HTML error page when an API route fails — which breaks any client trying to parse JSON.

---

## Files Involved

```bash
app/api/utils.py                # success_response and error_response helpers
app/api/routes.py               # Error handler registration
```

---

## Validation Checklist

* Every successful response includes `"success": true` and a `data` field
* Every error response includes `"success": false` and an `"error"` message
* Collection responses include a `"count"` field
* A request for a non-existent project slug returns `404` with a JSON body — not an HTML page
* No endpoint returns a bare object or array without the envelope

---

# 2.3 — Potential Front-End Decoupling

## Purpose

The API layer is not just a feature — it is a structural decision. By building a clean API that serves all project data, the portfolio's Jinja2 front end becomes one possible consumer of that data rather than the only one.

This section does not require building a React front end. It requires building the API in a way that would make decoupling possible without rewriting the back end.

---

## What Decoupling Would Look Like

In a decoupled architecture:

```text
Current (coupled):
Browser → Flask → Jinja2 templates → HTML response

Future (decoupled):
Browser → React SPA → Flask API → JSON response
```

The back end becomes a pure data service. The front end becomes a separate application that consumes it. They communicate only through the API contract.

---

## What This Section Prepares

To support future decoupling, the API must:

* return all data a front end would need to render every page — no reliance on server-side template context
* include media paths as full relative URLs, not bare filenames
* not require session state to return data — the API is stateless
* version its routes from the start (`/api/v1/`) so future changes do not break existing consumers

None of this requires extra work if the endpoints in 2.1 and 2.2 are implemented correctly. It is a mindset applied during implementation, not a separate task.

---

## CORS Configuration

If the front end is ever served from a different origin, the API will need CORS headers. Install and configure `flask-cors` in advance so this is not a blocker later:

```python
# app/app.py
from flask_cors import CORS
CORS(app, resources={r"/api/*": {"origins": "*"}})
```

This adds no overhead in the current architecture and costs nothing to include now.

---

## Files Involved

```bash
app/app.py                      # CORS registration
requirements.txt                # flask-cors added
```

---

## Validation Checklist

* Every piece of data shown in the Jinja2 templates is also available through the API
* Media paths are returned as usable relative URLs, not bare filenames
* API routes are versioned under `/api/v1/`
* CORS is configured for all `/api/*` routes
* A complete project listing page could be rendered using only the API response — no additional server calls required

---

# Validation

## API Functionality Test

Test all endpoints directly using a browser, curl, or a tool like Insomnia or Postman:

```bash
curl https://yourportfolio.com/api/v1/projects
curl https://yourportfolio.com/api/v1/projects/flask-portfolio
curl https://yourportfolio.com/api/v1/projects?category=web
curl https://yourportfolio.com/api/v1/projects?featured=true
curl https://yourportfolio.com/api/v1/categories
curl https://yourportfolio.com/api/v1/tags
```

Each request should return valid JSON with the correct envelope shape and status code.

## Error Handling Test

* Request a project with a slug that does not exist — confirm `404` with JSON body
* Confirm no endpoint returns an HTML error page under any condition

## Decoupling Readiness Test

Pick one public-facing page — the projects listing. Confirm that everything needed to render that page is present in the API response: titles, summaries, tech stack, category, tags, card image path, and GitHub link. If anything is missing from the response, it needs to be added to `to_dict()`.

---

# Files Involved

API blueprint:

```bash
app/api/__init__.py
app/api/routes.py
app/api/utils.py
```

Models updated:

```bash
app/models/project.py           # to_dict() method
app/models/category.py          # to_dict() method
app/models/tag.py               # to_dict() method
```

Application config:

```bash
app/app.py                      # Blueprint registration, CORS config
requirements.txt                # flask-cors
```

---

# Result

With Section 2 complete, the portfolio exposes a clean, versioned REST API that:

* returns all project data through structured, filterable endpoints
* uses a consistent JSON response envelope across all routes
* handles errors with JSON responses and correct status codes
* is structured to support front-end decoupling without back-end changes

The portfolio is no longer just a Flask application with Jinja2 templates — it is an application with a defined data layer, a defined presentation layer, and a clean boundary between them.

---

# Final Position

At this point, the project includes:

* **Phase 7 · Section 2 — Hosting and Deployment**
* **Phase 7 · Section 3 — Custom Domain and HTTPS Configuration**
* **Phase 7 · Section 4 — Analytics and SEO Foundations**
* **Phase 8 · Section 1 — Copywriting and Narrative**
* **Phase 8 · Section 2 — Proof Signals**
* **Phase 9 · Section 1 — Admin Dashboard**
* **Phase 9 · Section 2 — API Layer**

The portfolio now has a production-grade admin system and a public REST API. What remains — Section 3, DevOps upgrades — takes the infrastructure to the same standard.

---